In [4]:
#Load android feature selected dataset for modeling:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, f1_score, accuracy_score
from imblearn.over_sampling import SMOTE
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split


android_selected_features = pd.read_csv("../Processed_Data/Selected_FeaturesDatasets/android_SelectedFeatures.csv")

In [5]:
"stress" in android_selected_features.columns

True

In [6]:
#Preparing dataset for analysis:
#Select relevant columns for analysis:
y = android_selected_features['stress']
X = android_selected_features.drop(columns=['stress', 'uid', 'day'])  # Drop target, ID columns, and date
print("Final feature set columns:", X.columns)
print("Final feature set shape:", X.shape)  

Final feature set columns: Index(['Unnamed: 0', 'race_alaskan native/white', 'audio_amp_mean_ep_2',
       'act_in_vehicle_ep_0', 'race_american indian/alaska native',
       'light_mean_ep_3', 'race_american indian/white', 'sse3-4',
       'audio_amp_std_ep_2', 'pam', 'race_asian', 'race_black',
       'act_still_ep_3', 'race_more than one', 'phq4-1', 'phq4-2', 'gender',
       'phq4_score', 'race_other/hispanic', 'phq4-4', 'loc_self_dorm_dur',
       'sse3-1', 'sse3-3', 'race_white'],
      dtype='str')
Final feature set shape: (7256, 24)


In [7]:
#Splitting data into test and train sets to prevent data leakage:

#Column that identifies groups (participants):
group_col = 'uid'

#80/20 training testing split, with one testing group:
gss = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=android_selected_features[group_col]))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = android_selected_features[group_col].iloc[train_idx]

#Checking stress label distribution to ensure the groups are stratified:
print("Train distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest distribution:")
print(y_test.value_counts(normalize=True))

Train distribution:
stress
2.0    0.345873
3.0    0.253521
1.0    0.253165
4.0    0.110537
5.0    0.036905
Name: proportion, dtype: float64

Test distribution:
stress
3.0    0.295689
2.0    0.282332
1.0    0.239830
4.0    0.100182
5.0    0.081967
Name: proportion, dtype: float64


In [8]:
# Stratified Group K-Fold
skf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

**Random Forest Classifier**

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt',
    class_weight=None,
    random_state=42,
    n_jobs=-1
)

# CROSS-VALIDATION

skf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

RF_fold_f1 = []
RF_fold_acc = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train, groups=groups_train)):
    
    X_fold_train = X_train.iloc[train_idx]
    y_fold_train = y_train.iloc[train_idx]
    X_fold_val = X_train.iloc[val_idx]
    y_fold_val = y_train.iloc[val_idx]
    
    # Imputation
    imputer = SimpleImputer(strategy='median')
    X_fold_train_imputed = imputer.fit_transform(X_fold_train)
    X_fold_val_imputed = imputer.transform(X_fold_val)
    
    # SMOTE
    smote = SMOTE(random_state=42)
    X_resampled, y_resampled = smote.fit_resample(X_fold_train_imputed, y_fold_train)
    
    # Train
    rf_model.fit(X_resampled, y_resampled)
    
    # Validate
    val_preds = rf_model.predict(X_fold_val_imputed)
    
    f1 = f1_score(y_fold_val, val_preds, average='weighted')
    acc = accuracy_score(y_fold_val, val_preds)
    
    RF_fold_f1.append(f1)
    RF_fold_acc.append(acc)
    
    print(f"\nFold {fold+1}")
    print(f"F1: {f1:.4f}, Accuracy: {acc:.4f}")
    print("Classification Report:")
    print(classification_report(y_fold_val, val_preds))

print("\n=========================")
print(f"Average F1: {np.mean(RF_fold_f1):.4f}")
print(f"Average Accuracy: {np.mean(RF_fold_acc):.4f}")
print("=========================")


# FINAL TRAINING (FULL DATA)

final_imputer = SimpleImputer(strategy='median')
X_train_imputed = final_imputer.fit_transform(X_train)
X_test_imputed = final_imputer.transform(X_test)

# SMOTE on full training set
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train_imputed, y_train)

# Train final model
rf_model.fit(X_resampled, y_resampled)


# TEST SET EVALUATION

test_preds = rf_model.predict(X_test_imputed)
test_probs = rf_model.predict_proba(X_test_imputed)

print("TEST PERFORMANCE")

# Only if y_test exists
if 'y_test' in locals():
    test_f1 = f1_score(y_test, test_preds, average='weighted')
    test_acc = accuracy_score(y_test, test_preds)

    print(f"F1 Score: {test_f1:.4f}")
    print(f"Accuracy: {test_acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, test_preds))



Fold 1
F1: 0.4975, Accuracy: 0.5091
Classification Report:
              precision    recall  f1-score   support

         1.0       0.70      0.70      0.70       371
         2.0       0.45      0.61      0.52       370
         3.0       0.41      0.26      0.31       242
         4.0       0.25      0.23      0.24        96
         5.0       0.46      0.24      0.32        74

    accuracy                           0.51      1153
   macro avg       0.45      0.41      0.42      1153
weighted avg       0.51      0.51      0.50      1153


Fold 2
F1: 0.3281, Accuracy: 0.3386
Classification Report:
              precision    recall  f1-score   support

         1.0       0.36      0.53      0.43       287
         2.0       0.35      0.32      0.34       386
         3.0       0.32      0.20      0.24       326
         4.0       0.30      0.34      0.31       119
         5.0       0.24      0.26      0.25        31

    accuracy                           0.34      1149
   macro av

**XGBoost Classifier**

In [20]:

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softmax',
    num_class=5,
    random_state=42,
    n_jobs=-1,
    eval_metric='mlogloss'  # avoids warning
)

# CROSS-VALIDATION

skf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

XGB_fold_f1 = []
XGB_fold_acc = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train, groups=groups_train)):
    
    # Split (numeric only for XGB)
    X_fold_train = X_train.iloc[train_idx].select_dtypes(include=[np.number])
    y_fold_train = y_train.iloc[train_idx]
    X_fold_val = X_train.iloc[val_idx].select_dtypes(include=[np.number])
    y_fold_val = y_train.iloc[val_idx]
    
    # Impute
    imputer = SimpleImputer(strategy='median')
    X_fold_train = imputer.fit_transform(X_fold_train)
    X_fold_val = imputer.transform(X_fold_val)
    
    # Convert labels to 0-based
    y_fold_train = y_fold_train - 1
    y_fold_val_adj = y_fold_val - 1
    
    # SMOTE
    smote = SMOTE(random_state=42)
    X_resampled, y_resampled = smote.fit_resample(X_fold_train, y_fold_train)
    
    # Train
    xgb_model.fit(X_resampled, y_resampled)
    
    # Predict (already 0-based)
    val_preds = xgb_model.predict(X_fold_val)
    
    # Convert predictions back to original labels
    val_preds = val_preds + 1
    
    # Metrics (use original labels)
    f1 = f1_score(y_fold_val, val_preds, average='weighted')
    acc = accuracy_score(y_fold_val, val_preds)
    
    XGB_fold_f1.append(f1)
    XGB_fold_acc.append(acc)
    
    print(f"\nFold {fold+1}")
    print(f"F1: {f1:.4f}, Accuracy: {acc:.4f}")
    print("Classification Report:")
    print(classification_report(y_fold_val, val_preds))

print(f"Average F1: {np.mean(XGB_fold_f1):.4f}")
print(f"Average Accuracy: {np.mean(XGB_fold_acc):.4f}")


# FINAL TRAINING (FULL DATA)

# Keep only numeric columns
X_train_num = X_train.select_dtypes(include=[np.number])
X_test_num = X_test.select_dtypes(include=[np.number])

# Impute
final_imputer = SimpleImputer(strategy='median')
X_train_imputed = final_imputer.fit_transform(X_train_num)
X_test_imputed = final_imputer.transform(X_test_num)

# Convert labels to 0-based
y_train_adj = y_train - 1

# SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train_imputed, y_train_adj)

# Train final model
xgb_model.fit(X_resampled, y_resampled)


# TEST SET EVALUATION

test_preds = xgb_model.predict(X_test_imputed)

# Convert back to original labels
test_preds = test_preds + 1

print("TEST PERFORMANCE")

if 'y_test' in locals():
    test_f1 = f1_score(y_test, test_preds, average='weighted')
    test_acc = accuracy_score(y_test, test_preds)

    print(f"F1 Score: {test_f1:.4f}")
    print(f"Accuracy: {test_acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, test_preds))


Fold 1
F1: 0.4142, Accuracy: 0.4224
Classification Report:
              precision    recall  f1-score   support

         1.0       0.60      0.58      0.59       371
         2.0       0.36      0.49      0.42       370
         3.0       0.31      0.24      0.27       242
         4.0       0.25      0.19      0.21        96
         5.0       0.43      0.18      0.25        74

    accuracy                           0.42      1153
   macro avg       0.39      0.34      0.35      1153
weighted avg       0.42      0.42      0.41      1153


Fold 2
F1: 0.3214, Accuracy: 0.3473
Classification Report:
              precision    recall  f1-score   support

         1.0       0.36      0.47      0.41       287
         2.0       0.35      0.51      0.41       386
         3.0       0.36      0.13      0.19       326
         4.0       0.32      0.18      0.23       119
         5.0       0.22      0.13      0.16        31

    accuracy                           0.35      1149
   macro av

**Multinomial Logistic Regression**

In [24]:
lr_model = LogisticRegression(
    solver='lbfgs',
    max_iter=1000,
    class_weight=None,
    random_state=42,
)

# CROSS-VALIDATION

skf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

LR_fold_f1 = []
LR_fold_acc = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train, groups=groups_train)):
    
    # Split (numeric only)
    X_fold_train = X_train.iloc[train_idx].select_dtypes(include=[np.number])
    y_fold_train = y_train.iloc[train_idx]
    X_fold_val = X_train.iloc[val_idx].select_dtypes(include=[np.number])
    y_fold_val = y_train.iloc[val_idx]
    
    # Impute
    imputer = SimpleImputer(strategy='median')
    X_fold_train = imputer.fit_transform(X_fold_train)
    X_fold_val = imputer.transform(X_fold_val)
    
    # Scale (VERY important for LR)
    scaler = StandardScaler()
    X_fold_train = scaler.fit_transform(X_fold_train)
    X_fold_val = scaler.transform(X_fold_val)
    
    # SMOTE (after scaling ✔️)
    smote = SMOTE(random_state=42)
    X_resampled, y_resampled = smote.fit_resample(X_fold_train, y_fold_train)
    
    # Train
    lr_model.fit(X_resampled, y_resampled)
    
    # Predict
    val_preds = lr_model.predict(X_fold_val)
    
    # Metrics
    f1 = f1_score(y_fold_val, val_preds, average='weighted')
    acc = accuracy_score(y_fold_val, val_preds)
    
    LR_fold_f1.append(f1)
    LR_fold_acc.append(acc)
    
    print(f"\nFold {fold+1}")
    print(f"F1: {f1:.4f}, Accuracy: {acc:.4f}")
    print("Classification Report:")
    print(classification_report(y_fold_val, val_preds))

print(f"Average F1: {np.mean(LR_fold_f1):.4f}")
print(f"Average Accuracy: {np.mean(LR_fold_acc):.4f}")


# FINAL TRAINING (FULL DATA)

# Numeric only
X_train_num = X_train.select_dtypes(include=[np.number])
X_test_num = X_test.select_dtypes(include=[np.number])

# Impute
final_imputer = SimpleImputer(strategy='median')
X_train_imputed = final_imputer.fit_transform(X_train_num)
X_test_imputed = final_imputer.transform(X_test_num)

# Scale
final_scaler = StandardScaler()
X_train_scaled = final_scaler.fit_transform(X_train_imputed)
X_test_scaled = final_scaler.transform(X_test_imputed)

# SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train_scaled, y_train)

# Train final model
lr_model.fit(X_resampled, y_resampled)


# TEST SET EVALUATION

test_preds = lr_model.predict(X_test_scaled)
test_probs = lr_model.predict_proba(X_test_scaled)

print("TEST PERFORMANCE")

if 'y_test' in locals():
    test_f1 = f1_score(y_test, test_preds, average='weighted')
    test_acc = accuracy_score(y_test, test_preds)

    print(f"F1 Score: {test_f1:.4f}")
    print(f"Accuracy: {test_acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, test_preds))


Fold 1
F1: 0.4351, Accuracy: 0.4380
Classification Report:
              precision    recall  f1-score   support

         1.0       0.58      0.57      0.57       371
         2.0       0.39      0.46      0.42       370
         3.0       0.36      0.34      0.35       242
         4.0       0.36      0.22      0.27        96
         5.0       0.33      0.27      0.30        74

    accuracy                           0.44      1153
   macro avg       0.40      0.37      0.38      1153
weighted avg       0.44      0.44      0.44      1153


Fold 2
F1: 0.3304, Accuracy: 0.3412
Classification Report:
              precision    recall  f1-score   support

         1.0       0.40      0.55      0.46       287
         2.0       0.42      0.25      0.31       386
         3.0       0.28      0.19      0.23       326
         4.0       0.32      0.53      0.40       119
         5.0       0.13      0.39      0.20        31

    accuracy                           0.34      1149
   macro av

**Personlized Models**

In [ ]:
# checking distribution of stress labels across participants to understand class balance:
android_selected_features.groupby('uid')['stress'].value_counts(normalize=True)

counts = android_selected_features['uid'].value_counts()

print(counts.describe())
print("\nSmallest participants:")
print(counts.sort_values().head(10))

enough_data = counts[counts >= 30]
small_data = counts[counts < 30]
#Done to make sure that we have enough samples per participant 

print(f"Participants with >=30 samples: {len(enough_data)}")
print(f"Participants with <30 samples: {len(small_data)}")

count     33.000000
mean     219.878788
std      138.397922
min       12.000000
25%       80.000000
50%      208.000000
75%      338.000000
max      441.000000
Name: count, dtype: float64

Smallest participants:
uid
159    12
136    23
187    47
215    54
42     55
75     57
181    72
101    74
169    80
183    96
Name: count, dtype: int64
Participants with >=30 samples: 31
Participants with <30 samples: 2


In [13]:
#Random Forest personalized models for each participant 

unique_participants = android_selected_features['uid'].unique()

personalized_results = {}

for participant in unique_participants:
    
    participant_data = android_selected_features[
        android_selected_features['uid'] == participant
    ]
    
    X_participant = participant_data.drop(columns=['stress', 'uid', 'day'])
    y_participant = participant_data['stress']
    
    # Skip if too little data
    if len(participant_data) < 10:
        continue
    
    # Skip if only one class
    if y_participant.nunique() < 2:
        continue
    
    # Train-test split
    X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
        X_participant, y_participant, test_size=0.2, random_state=42
    )
    
    # Keep numeric features only
    X_train_p = X_train_p.select_dtypes(include=[np.number])
    X_test_p = X_test_p.select_dtypes(include=[np.number])
    
    # Impute missing values
    imputer = SimpleImputer(strategy='median')
    X_train_p = imputer.fit_transform(X_train_p)
    X_test_p = imputer.transform(X_test_p)
    
    # Random Forest model
    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features='sqrt',
        class_weight='balanced',  # helps imbalance per participant
        random_state=42,
        n_jobs=-1
    )
    
    # Train
    model.fit(X_train_p, y_train_p)
    
    # Predict
    preds = model.predict(X_test_p)
    
    # Metrics
    f1 = f1_score(y_test_p, preds, average='weighted')
    acc = accuracy_score(y_test_p, preds)
    
    personalized_results[participant] = {
        'f1': f1,
        'accuracy': acc
    }

# Averages
avg_f1 = np.mean([res['f1'] for res in personalized_results.values()])
avg_acc = np.mean([res['accuracy'] for res in personalized_results.values()])

print(f"\nAverage F1 across personalized RF models: {avg_f1:.4f}")
print(f"Average Accuracy across personalized RF models: {avg_acc:.4f}")

f1_scores = [res['f1'] for res in personalized_results.values()]
print("Min F1:", np.min(f1_scores))
print("Max F1:", np.max(f1_scores))

/Users/sethunair/Desktop/Winter 2026/619/BMEN619_StudentLife/.venv/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['gender']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/sethunair/Desktop/Winter 2026/619/BMEN619_StudentLife/.venv/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['gender']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/sethunair/Desktop/Winter 2026/619/BMEN619_StudentLife/.venv/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['light_mean_ep_3']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/sethunair/Desktop/Winter 2026/619/BMEN619_StudentLife/.venv/lib/python3.12/site-packages/sklearn/imput


Average F1 across personalized RF models: 0.5329
Average Accuracy across personalized RF models: 0.5472
Min F1: 0.2261904761904762
Max F1: 0.8800000000000001


In [16]:

# Creating personalized XGBoost models for each participant
unique_participants = android_selected_features['uid'].unique()
personalized_results = {}
skipped_participants = []
processed_participants = []



for participant in unique_participants:
    
    participant_data = android_selected_features[android_selected_features['uid'] == participant]
    
    X_participant = participant_data.drop(columns=['stress', 'uid', 'day'])
    y_participant = participant_data['stress']
    
    if len(participant_data) < 10:
        skipped_participants.append((participant, "too few samples"))
        continue
    
    if y_participant.nunique() < 2:
        skipped_participants.append((participant, "only one class"))
        continue
    
    processed_participants.append(participant)
    
    # Train-test split
    X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
        X_participant, y_participant, test_size=0.2, random_state=42
    )
    
    # Keep numeric only
    X_train_p = X_train_p.select_dtypes(include=[np.number])
    X_test_p = X_test_p.select_dtypes(include=[np.number])
    
    # Impute
    imputer = SimpleImputer(strategy='median')
    X_train_p = imputer.fit_transform(X_train_p)
    X_test_p = imputer.transform(X_test_p)
    
    # Adjust labels to start at 0 for XGBoost classification
    y_train_adj = y_train_p - 1
    y_test_adj = y_test_p - 1
    
    # Model
    model = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='multi:softmax',
        num_class=len(np.unique(y_train_adj)),
        random_state=42,
        n_jobs=-1
    )
    
    model.fit(X_train_p, y_train_adj)
    preds = model.predict(X_test_p)
    
    # Metrics
    f1 = f1_score(y_test_adj, preds, average='weighted')
    acc = accuracy_score(y_test_adj, preds)
    
    personalized_results[participant] = {
        'f1': f1,
        'accuracy': acc
    }

# Average performance
avg_f1 = np.mean([res['f1'] for res in personalized_results.values()])
avg_acc = np.mean([res['accuracy'] for res in personalized_results.values()])
print(f"Total participants: {len(unique_participants)}")
print(f"Processed participants: {len(processed_participants)}")
print(f"Skipped participants: {len(skipped_participants)}")

print("\nSkip reasons:")
for p, reason in skipped_participants[:10]:  # show first 10
    print(p, "-", reason)

print(f"\nAverage F1 across personalized models: {avg_f1:.4f}")
print(f"Average Accuracy across personalized models: {avg_acc:.4f}")
f1_scores = [res['f1'] for res in personalized_results.values()]
print("Min F1:", np.min(f1_scores))
print("Max F1:", np.max(f1_scores))

/Users/sethunair/Desktop/Winter 2026/619/BMEN619_StudentLife/.venv/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['gender']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/sethunair/Desktop/Winter 2026/619/BMEN619_StudentLife/.venv/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['gender']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/sethunair/Desktop/Winter 2026/619/BMEN619_StudentLife/.venv/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['light_mean_ep_3']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/sethunair/Desktop/Winter 2026/619/BMEN619_StudentLife/.venv/lib/python3.12/site-packages/sklearn/imput

Total participants: 33
Processed participants: 33
Skipped participants: 0

Skip reasons:

Average F1 across personalized models: 0.5188
Average Accuracy across personalized models: 0.5349
Min F1: 0.30634920634920637
Max F1: 0.8289650537634409


In [15]:
#Multinomial Logistic Regression personalized models for each participant

unique_participants = android_selected_features['uid'].unique()

personalized_lr_results = {}

for participant in unique_participants:
    
    participant_data = android_selected_features[
        android_selected_features['uid'] == participant
    ]
    
    X_participant = participant_data.drop(columns=['stress', 'uid', 'day'])
    y_participant = participant_data['stress']
    
    # Skip checks
    if len(participant_data) < 10:
        continue
    
    if y_participant.nunique() < 2:
        continue
    
    # Split
    X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
        X_participant, y_participant, test_size=0.2, random_state=42
    )
    
    # Numeric only
    X_train_p = X_train_p.select_dtypes(include=[np.number])
    X_test_p = X_test_p.select_dtypes(include=[np.number])
    
    # Impute
    imputer = SimpleImputer(strategy='median')
    X_train_p = imputer.fit_transform(X_train_p)
    X_test_p = imputer.transform(X_test_p)
    
    #  Scale 
    scaler = StandardScaler()
    X_train_p = scaler.fit_transform(X_train_p)
    X_test_p = scaler.transform(X_test_p)
    
    # Model
    model = LogisticRegression(
        solver='lbfgs',
        max_iter=1000,
        class_weight='balanced',  # helps within-user imbalance
        random_state=42
    )
    
    # Train
    model.fit(X_train_p, y_train_p)
    
    # Predict
    preds = model.predict(X_test_p)
    
    # Metrics
    f1 = f1_score(y_test_p, preds, average='weighted')
    acc = accuracy_score(y_test_p, preds)
    
    personalized_lr_results[participant] = {
        'f1': f1,
        'accuracy': acc
    }

# Averages
avg_f1 = np.mean([res['f1'] for res in personalized_lr_results.values()])
avg_acc = np.mean([res['accuracy'] for res in personalized_lr_results.values()])

print(f"\nAverage F1 across personalized LR models: {avg_f1:.4f}")
print(f"Average Accuracy across personalized LR models: {avg_acc:.4f}")

#  variability
f1_scores = [res['f1'] for res in personalized_lr_results.values()]
print("Min F1:", np.min(f1_scores))
print("Max F1:", np.max(f1_scores))

/Users/sethunair/Desktop/Winter 2026/619/BMEN619_StudentLife/.venv/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['gender']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/sethunair/Desktop/Winter 2026/619/BMEN619_StudentLife/.venv/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['gender']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(



Average F1 across personalized LR models: 0.4648
Average Accuracy across personalized LR models: 0.4538
Min F1: 0.08333333333333333
Max F1: 0.7466666666666667


/Users/sethunair/Desktop/Winter 2026/619/BMEN619_StudentLife/.venv/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['light_mean_ep_3']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/sethunair/Desktop/Winter 2026/619/BMEN619_StudentLife/.venv/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['light_mean_ep_3']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
